In [1]:
import numpy as np
import torch
import pandas as pd
from helper import Autoencoder, load_data, train, save_params

In [2]:
known_strengths = {'null':10,'N4': 0.0, 'Q4': 1.3340727612197436, 'Q7': 2.428134794028789, 'T4': 1.9599578912997808, 'V4': 3.2307473950102388, 'G4': 4.514668716435611, 'E1': 5.21564553829087, 'A2': 0.43209185328878835, 'Y3': 0.8603530802315512}
train_strength = True

In [3]:
# Parameters for the autoencoder that can be changed

ignore_indexes=['Event','Replicate'] #ensure at least Event is ignored so that it can group properly since it is unique to each cell
group_size=100
batch_size=100
reconstruction_weight=1
strength_weight=0.0001
num_epochs=15
embedding_size=2
autoencoder_hidden_sizes=[250,200]
train_test_split=0.8
model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD38', 'CD4', 'CD44', 'CD45',
       'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
       'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
       'Proliferation', 'SSC-A', 'TBet']
# Try without CD38 (combined dataset has no CD38)
# model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD4', 'CD44', 'CD45',
#        'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
#        'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
#        'Proliferation', 'SSC-A', 'TBet']
model_path = "autoencoder_test.pt"
data_path = "../../initialSingleCellDf-channel-20220916-MW_018-001.h5"
# data_path = "../../initialSingleCellDf-channel-20220926-MW_020.h5"

def get_strength(labels,data_index_names):
       if 'Peptide' not in data_index_names:
              print("No peptide column found")
              return -1
       antigen_column = list(data_index_names).index('Peptide')
       antigen = labels[antigen_column]
       if antigen in known_strengths:
              return known_strengths[antigen]
       else:
              print("Antigen not found: ",antigen)
              return -1

In [4]:
data = pd.read_hdf(data_path, key="df")
# Add any filters here to remove data that you don't want the model to be trained on
data = data.loc[(data.index.get_level_values('CellType') == 'OT-1') & ((data.index.get_level_values('Time') >= 40) | (data.index.get_level_values('Peptide') == 'null'))]#& (data.index.get_level_values('Peptide') != 'T4')]

In [5]:
dataset, index_order, data_index_names, data_columns, data_labels,missing_columns = load_data(data,model_inputs,ignore_indexes,group_size,transform='normalize')
print("Missing columns: ",missing_columns)

Missing columns:  []


In [6]:
# add to the dataset the known strengths for each sample

def add_column_to_label(dataset,new_column):
    data = []
    labels = []
    for i in range(len(dataset)):
        new_labels = np.append(dataset[i][1],new_column[i])
        data.append(np.array(dataset[i][0]))
        labels.append(new_labels)
    data = np.array(data)
    labels = np.array(labels, dtype=np.float32)  # Convert labels to float32
    new_dataset = torch.utils.data.TensorDataset(torch.tensor(data),torch.tensor(labels))
    return new_dataset, len(labels[0])-1

strengths = [get_strength(labels,data_index_names) for labels in data_labels]
new_dataset, strength_index = add_column_to_label(dataset,strengths)


In [7]:
train_size = int(train_test_split * len(new_dataset))
test_size = len(new_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(new_dataset, [train_size, test_size])
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = Autoencoder(len(model_inputs),autoencoder_hidden_sizes,embedding_size).to(device)

In [9]:
train(model,train_loader,test_loader,reconstruction_weight,strength_weight,strength_index,device,num_epochs=num_epochs,train_strength=train_strength)
torch.save(model.state_dict(), model_path)
save_params(model_inputs,group_size,batch_size,embedding_size,autoencoder_hidden_sizes,model_path)

epoch [1/15], train loss:1.304852 val loss:0.584096 val reconstruction loss 0.166920 val strength loss 0.834351
epoch [2/15], train loss:0.492665 val loss:0.418312 val reconstruction loss 0.141095 val strength loss 0.554433
epoch [3/15], train loss:0.381001 val loss:0.364347 val reconstruction loss 0.129849 val strength loss 0.468997
epoch [4/15], train loss:0.333980 val loss:0.340785 val reconstruction loss 0.118965 val strength loss 0.443641
epoch [5/15], train loss:0.307675 val loss:0.289853 val reconstruction loss 0.114334 val strength loss 0.351038
epoch [6/15], train loss:0.287631 val loss:0.288250 val reconstruction loss 0.113571 val strength loss 0.349358
epoch [7/15], train loss:0.268918 val loss:0.288894 val reconstruction loss 0.109703 val strength loss 0.358382
epoch [8/15], train loss:0.259954 val loss:0.276964 val reconstruction loss 0.113696 val strength loss 0.326537
epoch [9/15], train loss:0.250365 val loss:0.333618 val reconstruction loss 0.104925 val strength loss 0